# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. All references to record sets, fields, and columns use their `@id` for reproducibility and clarity.

### Dataset Source
The dataset source is provided via a Croissant schema URL and is intended for examples of exploration, processing, and FAIR analysis.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Dataset License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s in the dataset.

In [ ]:
# Explore the dataset structure
print("Available Record Sets (by @id):")
for recset in dataset.record_sets:
    print(f"  RecordSet name: {recset.name}")
    print(f"    @id: {recset.id}")
    print(f"    Description: {getattr(recset, 'description', '')}")
    print("    Fields:")
    for field in recset.fields:
        print(f"      - Field name: {field.name}, @id: {field.id}, dataType: {getattr(field, 'data_type', None)}")
    print()

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. All entities (record set, field, column) are referenced **by their `@id`**.

We will extract all main record sets. You can modify the selection based on your needs.

In [ ]:
# Collect all record set @ids
record_set_ids = [r.id for r in dataset.record_sets]
print("All available RecordSet @ids:")
for rid in record_set_ids:
    print(f"  - {rid}")

# Extract records for each record set into a DataFrame
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    if records:
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded {len(df)} records for RecordSet @id {rsid}")
    else:
        print(f"No records found for RecordSet @id {rsid}")

# Display columns for a selected (main data) record set
if record_set_ids:
    main_rs = record_set_ids[0]
    print(f"Columns in record set {main_rs}:\n", dataframes[main_rs].columns.tolist())
    display(dataframes[main_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. We select a numeric field (e.g., GAD-7 score), filter, normalize, and group data by demographic attribute. All references use their `@id`s.

**Please replace the record set and field `@id`s with those relevant for your own analysis, as shown in the overview above.**

In [ ]:
# Example: choose a record set containing survey responses
# (Replace with the @id of the main records set if different in your dataset)
rs_id = main_rs  # Example - set appropriately
df = dataframes[rs_id]

# Inspect columns to find a numeric field (using @id).
print("All columns in selected record set:")
print(df.columns.tolist())

# ---- Set these IDs based on actual field @ids from your dataset schema ---- #
# Example field names: replace with the actual @id (or column name) of a numeric field
gad7_field = None
for c in df.columns:
    if 'gad7' in c.lower() or 'GAD7' in c:
        gad7_field = c
        break
if gad7_field is None:
    # Pick first numeric-looking field if possible
    num_cols = df.select_dtypes(include='number').columns.tolist()
    gad7_field = num_cols[0] if num_cols else df.columns[0]
print(f"Selected numeric field for analysis: {gad7_field}")

threshold = df[gad7_field].mean() if pd.api.types.is_numeric_dtype(df[gad7_field]) else None
if threshold is not None:
    filtered_df = df[df[gad7_field] > threshold]
    print(f"Filtered records with {gad7_field} > {threshold:.2f}:")
    display(filtered_df.head())
    filtered_df[f"{gad7_field}_normalized"] = (filtered_df[gad7_field] - filtered_df[gad7_field].mean()) / filtered_df[gad7_field].std()
    print(f"Normalized {gad7_field} for filtered records:")
    display(filtered_df[[gad7_field, f"{gad7_field}_normalized"]].head())
    # Grouping by a demographic attribute (replace with appropriate field @id)
    group_field = None
    for cand in ['level_of_education', 'village', 'age_group', 'gender', 'marital_status']:
        for c in df.columns:
            if cand in c.lower():
                group_field = c
                break
        if group_field:
            break
    print(f"Grouping by: {group_field if group_field else 'none found'}")
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[gad7_field].mean().reset_index()
        print(f"Mean {gad7_field} grouped by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields, using pandas and matplotlib/seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for GAD-7 scores (or selected numeric field)
if gad7_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[gad7_field].dropna(), kde=True)
    plt.title(f"Distribution of {gad7_field}")
    plt.xlabel(gad7_field)
    plt.ylabel("Count")
    plt.show()

# Boxplot by group field if available
if group_field and group_field in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=gad7_field, data=df)
    plt.title(f"{gad7_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, extract, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. All references to data entities were made using their `@id`s as per Croissant schema best practices, ensuring reproducible and auditable results. Further analysis can extend these steps for downstream statistical modeling or visualization.